# TPCx-AI UC8 — Databricks Data Load

Loads ORDERS / LINEITEM / PRODUCT for scale factors 1, 10, 100, 1000 into Hive metastore Delta tables (`TPCXAI_V2_SF{N}.{ORDERS,LINEITEM,PRODUCT}`), ready for `uc8_xgboost_benchmark_dbx.ipynb`. `df.write.saveAsTable(...)` writes Delta by default on Databricks Runtime for Machine Learning.

## Prerequisites

1. Generate UC8 CSVs with the official [TPCx-AI kit](https://www.tpc.org/tpcx-ai/). For each scale factor you want, you need `order.csv`, `lineitem.csv`, `product.csv`.
2. Upload them to your S3 bucket under `s3a://YOUR_BUCKET/tpcxai/uc08/sf<N>/`.
3. Replace `YOUR_BUCKET`, `YOUR_AWS_KEY_ID`, `YOUR_AWS_SECRET_KEY` in cell 1 below. For production, prefer an instance profile or Unity Catalog external location over inline keys.
4. Attach this notebook to a cluster running Databricks Runtime for Machine Learning and run all cells. To skip a scale factor, drop it from the `[1, 10, 100, 1000]` list in cell 2.

The final cell creates `TPCXAI_V2_RESULTS`, the database where the benchmark notebook appends per-run timings.


In [ ]:
# S3 credentials (one-time per cluster session). Replace placeholders before running.
spark.conf.set("fs.s3a.access.key", "YOUR_AWS_KEY_ID")
spark.conf.set("fs.s3a.secret.key", "YOUR_AWS_SECRET_KEY")

# Sanity-check the bucket is reachable.
dbutils.fs.ls("s3a://YOUR_BUCKET/")


In [ ]:
S3_BASE = "s3a://YOUR_BUCKET/tpcxai/uc08"

# Hive metastore: flat namespace, use TPCXAI_V2_SF1 etc.
for sf in [1, 10, 100, 1000]:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS TPCXAI_V2_SF{sf}")

# Table schemas
from pyspark.sql.types import *

order_schema = StructType([
    StructField("O_ORDER_ID", IntegerType()),
    StructField("O_CUSTOMER_SK", IntegerType()),
    StructField("WEEKDAY", StringType()),
    StructField("DATE", StringType()),
    StructField("STORE", IntegerType()),
    StructField("TRIP_TYPE", IntegerType()),
])

lineitem_schema = StructType([
    StructField("LI_ORDER_ID", IntegerType()),
    StructField("LI_PRODUCT_ID", IntegerType()),
    StructField("QUANTITY", IntegerType()),
    StructField("PRICE", DoubleType()),
])

product_schema = StructType([
    StructField("P_PRODUCT_ID", IntegerType()),
    StructField("NAME", StringType()),
    StructField("DEPARTMENT", StringType()),
])

tables = {
    "ORDERS":   ("order.csv",    order_schema),
    "LINEITEM": ("lineitem.csv", lineitem_schema),
    "PRODUCT":  ("product.csv",  product_schema),
}

# Load all SFs (saveAsTable defaults to Delta on Databricks Runtime for Machine Learning).
for sf in [1, 10, 100, 1000]:
    print(f"\n{'='*40} SF={sf} {'='*40}")
    for table_name, (csv_file, schema) in tables.items():
        path = f"{S3_BASE}/sf{sf}/{csv_file}"
        dest = f"TPCXAI_V2_SF{sf}.{table_name}"
        print(f"  Loading {path} -> {dest} ...", end=" ")
        df = spark.read.csv(path, header=True, schema=schema)
        df.write.mode("overwrite").saveAsTable(dest)
        cnt = spark.table(dest).count()
        print(f"{cnt:,} rows")

print("\nDone!")


In [ ]:
# Database for benchmark output (RESULT_TABLE in uc8_xgboost_benchmark_dbx.ipynb).
spark.sql("CREATE DATABASE IF NOT EXISTS TPCXAI_V2_RESULTS")
